# 02 · Functions & Scope

`*args`/`**kwargs`, the mutable-default trap, LEGB scope and `nonlocal`, closures, lambda/map/filter/reduce. Closures here are the foundation for decorators in notebook 07 — don't skip them.

### 2.1 — `*args`

Write `total(*args)` that returns the sum of all positional arguments passed, however many.

```
total(1, 2, 3) → 6
total() → 0
```

In [1]:
def total(*args):
    return sum(args)

# Tests
assert total(1, 2, 3) == 6
assert total() == 0
assert total(10) == 10
print("2.1 passed")

2.1 passed


### 2.2 — `**kwargs`

Write `describe(**kwargs)` that returns a list of `"key=value"` strings, sorted alphabetically by key.

```
describe(b=2, a=1) → ["a=1", "b=2"]
```

In [4]:
def describe(**kwargs):
    return [f"{k}={v}" for k, v in sorted(kwargs.items())]

# Tests
assert describe(b=2, a=1) == ["a=1", "b=2"]
assert describe() == []
assert describe(x=5) == ["x=5"]
print("2.2 passed")

2.2 passed


### 2.3 — The mutable default argument trap

This is the most infamous gotcha in Python. A default argument is evaluated **once**, when the 
function is defined — not each call. So a mutable default (like `[]`) is **shared across all calls**:

```python
def broken(x, acc=[]):      # acc created ONCE, reused forever
    acc.append(x)
    return acc

broken(1)   # [1]
broken(2)   # [1, 2]  ← the SAME list, still holding 1!
```

The fix: use `None` as the default and create a fresh list inside. Write `accumulate(x, acc=None)` 
correctly so each call with no `acc` starts fresh.

In [1]:
def accumulate(x, acc=None):
    if acc is None:
        acc = []
    acc.append(x)
    return acc

# Tests
assert accumulate(1) == [1]
assert accumulate(2) == [2]      # fresh — NOT [1, 2]
existing = [10]
assert accumulate(20, existing) == [10, 20]
print("2.3 passed")

2.3 passed


### 2.4 — Scope and the LEGB rule

Python resolves a name by searching **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in, in that 
order. Assigning to a name inside a function makes it local *unless* you declare otherwise.

`nonlocal` lets an inner function rebind a variable in the **enclosing** function. Write `make_counter` 
that returns a function which increments and returns a count each time it is called.

```
c = make_counter()
c() → 1
c() → 2
```

You'll need `nonlocal` to mutate the enclosing `count`.

In [7]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

# Tests
c = make_counter()
assert c() == 1
assert c() == 2
assert c() == 3
c2 = make_counter()       # independent
assert c2() == 1
print("2.4 passed")

2.4 passed


### 2.5 — Closures: functions that capture state

`make_counter` above was already a closure — `increment` "closed over" `count`. A closure is a 
function plus the enclosing variables it remembers, even after the outer function has returned.

Write `multiplier(n)` that returns a function multiplying its argument by `n`. The returned function 
must remember `n`.

```
double = multiplier(2)
double(5) → 10
triple = multiplier(3)
triple(5) → 15
```

In [8]:
def multiplier(n):

    def multiply(x):
        return x*n

    return multiply

# Tests
double = multiplier(2)
triple = multiplier(3)
assert double(5) == 10
assert triple(5) == 15
assert double(100) == 200   # double still remembers n=2
print("2.5 passed")

2.5 passed


### 2.6 — lambda, map, filter

`lambda` is an anonymous one-expression function. `map(f, iterable)` applies `f` to each element; 
`filter(f, iterable)` keeps elements where `f` is truthy. Both return lazy iterators — wrap in `list()`.

Using `map`, `filter`, and `lambda` (no comprehension), return the squares of the even numbers.

```
even_squares([1,2,3,4,5,6]) → [4, 16, 36]
```

In [9]:
def even_squares(nums):
    return list(map(lambda x: x*x, filter(lambda x: x%2==0, nums)))

# Tests
assert even_squares([1, 2, 3, 4, 5, 6]) == [4, 16, 36]
assert even_squares([1, 3, 5]) == []
print("2.6 passed")

2.6 passed


### 2.7 — `reduce`

`functools.reduce(f, iterable, initial)` folds an iterable into a single value by repeatedly applying 
a two-argument function. `reduce(f, [a,b,c], init)` computes `f(f(f(init, a), b), c)`.

Using `reduce`, compute the product of a list of numbers (empty list → 1).

```
product([1,2,3,4]) → 24
```

In [11]:
from functools import reduce

def product(nums):
    if not nums:
        return 1
    return reduce(lambda x, y: x*y, nums)

# Tests
assert product([1, 2, 3, 4]) == 24
assert product([]) == 1
assert product([5]) == 5
print("2.7 passed")

2.7 passed
